# 参数配置


In [ ]:
import os


# data path
BASE_DATA_DIR = "./68 model"  # 数据根目录
EXPORTED_ANNO_DIR = "exported_coco17_strict1"  # COCO格式JSON所在子目录
VIEW_NAMES = [f"{i:02d} visual_view" for i in range(1, 6)]
print(VIEW_NAMES)

# 图像文件名模板
IMG_NAME_TEMPLATES = ["{:04d}" + f"_{view_name[1]}.png" for view_name in VIEW_NAMES]
print(f"图像文件名模板: {IMG_NAME_TEMPLATES}")

# output
COCO_JSON_PREFIX = "./anime_coco"  # 生成的COCO标注前缀
WORK_DIR = "./work_dirs/anime_rtmpose_tiny"  # 训练工作目录
TRAIN_VAL_SPLIT = 0.85

# models (RTMPose family)
MODEL_CONFIG_NAME = "rtmpose-t_8xb256-420e_coco-256x192.py"

RTMPOSE_MODEL_CONFIG = f"./mmpose/configs/body_2d_keypoint/rtmpose/coco/{MODEL_CONFIG_NAME}"


# 基础模型权重路径（用于对比baseline）
BASELINE_CHECKPOINT = RTMPOSE_MODEL_CONFIG


# 训练超参数
MAX_EPOCHS = 30
BATCH_SIZE = 8
LR = 5e-4
VAL_INTERVAL = 5
INPUT_SIZE = (192, 256)
DEVICE = 'cuda:0'
RANDOM_SEED = 42

# COCO 17关键点标准顺序
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# 骨骼连接（用于可视化）
SKELETON = [
    [15,13],[13,11],[16,14],[14,12],[11,12],
    [5,11],[6,12],[5,6],[5,7],[6,8],
    [7,9],[8,10],[1,2],[0,1],[0,2],
    [1,3],[2,4],[3,5],[4,6]
]

In [ ]:

import sys
import json
import glob
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

# MMPose setup
if os.path.exists('./mmpose'):
    sys.path.insert(0, os.path.abspath('./mmpose'))
else:
    print("mmpose directory not found.")

from mmengine import Config
from mmengine.runner import Runner
from mmpose.utils import register_all_modules
from mmpose.datasets import CocoDataset
from PIL import Image
from matplotlib import pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.patches as mpatches

register_all_modules()
print("MMPose version check passed.")

# 转换为标准COCO格式
工具函数已解耦到 `coco_converter.py`，此处仅做调用。
- train: `./anime_coco_train.json`
- validation: `./anime_coco_val.json`

In [ ]:
from coco_converter import convert_and_split_data

train_anno_file, val_anno_file = convert_and_split_data(
    model_dir      = BASE_DATA_DIR,
    exported_subdir  = EXPORTED_ANNO_DIR,
    # view_names         = VIEW_NAMES,
    # img_name_templates = IMG_NAME_TEMPLATES,
    keypoint_names     = KEYPOINT_NAMES,
    coco_json_prefix   = COCO_JSON_PREFIX,
    train_val_split    = TRAIN_VAL_SPLIT,
    random_seed        = RANDOM_SEED,
)
print(f"训练集: {train_anno_file}")
print(f"验证集: {val_anno_file}")

## 转换后骨骼可视化预览
使用 `ipywidgets` 实现：
- 切换训练集 / 验证集
- 帧索引滑块
- 显示关键点 + 骨骼连接

可视化模块在设计上由三层构成：


1. 数据层 _load_coco_index() 把 COCO JSON 加载后建两个字典，img_map 和 ann_map 都以 image_id 为 key，后续按 id 查找是 O(1) 的，不需要每次遍历整个列表。
2. 绘图层 _draw_sample() 是核心，接受一个 matplotlib ax 子图对象和一条样本数据，依次做三件事：用 PIL 打开图像再 imshow 渲染为背景；用 mpatches.Rectangle 在 bbox 坐标处画绿框；遍历 COCO17_SKELETON_0IDX（0-indexed 骨架连线），对每条边检查两端点 visibility，都大于 0 才画线，最后把 visibility=2 的点画红色散点，标签用 ax.text 贴在点旁边并加白色半透明背景防止与图像混淆。
3. 交互层 _make_validator_widget() 用 ipywidgets 的 ToggleButtons、IntSlider、Checkbox 组成控件面板，通过 widgets.interactive_output 把控件值绑定到 _run 函数，每次控件变化自动重新调用 visualize_coco_samples()。VBox/HBox 是纯布局容器，控制控件的排列方向。

入口函数 visualize_coco_samples() 负责按随机种子抽样 id，用 plt.subplots(rows, cols) 创建网格画布，把每个 id 分配给对应的 ax 传给 _draw_sample，多余的子图用 axis("off") 隐藏。

In [ ]:

# arguments for visualization
TRAIN_JSON     = Path(COCO_JSON_PREFIX +"_train.json")         # convert_and_split_data 返回的 train_path
VAL_JSON       = Path(COCO_JSON_PREFIX + "_val.json")           # convert_and_split_data 返回的 val_path
CHECK_IMG_PATH = False  # 是否打印图像abspath, debug


COCO17_SKELETON_0IDX = [     # 0-indexed，用于可视化连线
    (15, 13), (13, 11), (16, 14), (14, 12), (11, 12),
    (5, 11),  (6, 12),  (5, 6),   (5, 7),   (6, 8),
    (7, 9),   (8, 10),  (1, 2),   (0, 1),   (0, 2),
    (1, 3),   (2, 4),   (3, 5),   (4, 6),
]

LIMB_COLORS = [
    "#FF6B6B", "#FF6B6B", "#4ECDC4", "#4ECDC4", "#45B7D1",
    "#96CEB4", "#96CEB4", "#FFEAA7", "#DDA0DD", "#DDA0DD",
    "#DDA0DD", "#DDA0DD", "#FFB347", "#FFB347", "#FFB347",
    "#87CEEB", "#87CEEB", "#87CEEB", "#87CEEB",
]

KPT_COLOR_VISIBLE   = "#FF4500"   # visibility=2
KPT_COLOR_INVISIBLE = "#999999"   # visibility=0（画面外，通常不显示）


def _load_coco_index(json_path: str) -> dict:
    """读取 COCO JSON，构建 image_id -> (image_info, annotation) 索引。"""
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    img_map = {img["id"]: img for img in coco["images"]}
    ann_map = {ann["image_id"]: ann for ann in coco["annotations"]}
    ids = sorted(img_map.keys())
    return {"img": img_map, "ann": ann_map, "ids": ids, "meta": coco.get("categories", [])}


def _draw_sample(ax, base_dir: str, img_info: dict, ann: dict,
                 show_invisible: bool = False, show_labels: bool = True, 
                 check_img_path: bool = CHECK_IMG_PATH):
    """在 ax 上绘制单张样本的图像、bbox &关键点"""
    img_path = Path(base_dir) / img_info["file_name"]
    if check_img_path: print(Path.absolute(img_path))

    try:
        img = np.array(Image.open(img_path).convert("RGB"))
        ax.imshow(img)
    except Exception as e:
        ax.set_facecolor("#222")
        ax.text(0.5, 0.5, f"图像加载失败\n{e}", color="white",
                ha="center", va="center", transform=ax.transAxes, fontsize=9)

    # BBox
    bbox = ann.get("bbox")
    if bbox and bbox[2] > 0 and bbox[3] > 0:
        rect = mpatches.Rectangle(
            (bbox[0], bbox[1]), bbox[2], bbox[3],
            linewidth=2, edgecolor="#00FF7F", facecolor="none", zorder=4
        )
        ax.add_patch(rect)

    # 关键点
    kps = ann.get("keypoints", [])
    num_kp = len(kps) // 3
    pts = []
    for i in range(num_kp):
        x, y, v = kps[i*3], kps[i*3+1], kps[i*3+2]
        pts.append((x, y, int(v)))

    # 骨架连线（仅连 visibility>0 的点）
    for idx, (a, b) in enumerate(COCO17_SKELETON_0IDX):
        if a < len(pts) and b < len(pts):
            xa, ya, va = pts[a]
            xb, yb, vb = pts[b]
            if va > 0 and vb > 0:
                color = LIMB_COLORS[idx] if idx < len(LIMB_COLORS) else "#FFFFFF"
                ax.plot([xa, xb], [ya, yb], color=color, linewidth=2, zorder=3)

    # 散点
    COCO17_NAMES = [
        "nose", "l_eye", "r_eye", "l_ear", "r_ear",
        "l_sho", "r_sho", "l_elb", "r_elb", "l_wri", "r_wri",
        "l_hip", "r_hip", "l_kne", "r_kne", "l_ank", "r_ank",
    ]
    for i, (x, y, v) in enumerate(pts):
        if v == 2:
            ax.scatter(x, y, s=40, c=KPT_COLOR_VISIBLE, zorder=5)
            if show_labels:
                label = COCO17_NAMES[i] if i < len(COCO17_NAMES) else str(i)
                ax.text(x + 4, y - 4, label, fontsize=7, color=KPT_COLOR_VISIBLE,
                        bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.5), zorder=6)
        elif v == 0 and show_invisible:
            ax.scatter(x, y, s=20, c=KPT_COLOR_INVISIBLE, marker="x", zorder=5)

    n_vis = sum(1 for _, _, v in pts if v == 2)
    ax.set_title(
        f"id={img_info['id']}  vis={n_vis}/{num_kp}\n{Path(img_info['file_name']).name}",
        fontsize=9
    )
    ax.axis("off")


def visualize_coco_samples(
    json_path: str,
    base_dir: str,
    num_samples: int = 6,
    cols: int = 3,
    random_seed: int = 0,
    show_invisible: bool = False,
    show_labels: bool = True,
    specific_ids: list = None,
):
    """
    可视化 COCO JSON 中若干样本。

    Parameters
    ----------
    json_path       : COCO JSON 路径（train 或 val）
    base_dir        : 图像根目录（file_name 的前缀目录）
    num_samples     : 随机抽取数量
    cols            : 每行列数
    random_seed     : 随机种子
    show_invisible  : 是否显示 visibility=0 的点（灰叉）
    show_labels     : 是否显示关节名称
    specific_ids    : 若指定，则只显示这些 image_id
    """
    idx = _load_coco_index(json_path)
    ids = idx["ids"]

    if specific_ids:
        ids = [i for i in specific_ids if i in idx["img"]]
    else:
        random.seed(random_seed)
        ids = random.sample(ids, min(num_samples, len(ids)))

    rows = (len(ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 6))
    axes = np.array(axes).flatten()

    for ax_i, img_id in enumerate(ids):
        img_info = idx["img"][img_id]
        ann = idx["ann"].get(img_id, {})
        _draw_sample(axes[ax_i], base_dir, img_info, ann,
                     show_invisible=show_invisible, show_labels=show_labels)

    for ax_i in range(len(ids), len(axes)):
        axes[ax_i].axis("off")

    split_tag = "TRAIN" if "train" in Path(json_path).name else "VAL"
    fig.suptitle(
        f"[{split_tag}] {json_path}\n"
        f"{len(ids)} samples;  green=bbox  red=keypoints(VISIBLE)  gray=x(invisible)",
        fontsize=11, y=1.01
    )
    plt.tight_layout()
    plt.show()
    print(f"number of samples in this JSON {len(idx['ids'])}")


# widget UI 
def _make_validator_widget(base_dir: str, train_json: str, val_json: str):
    # split_toggle  = widgets.ToggleButtons(options=["train", "val"], description="集合")
    # seed_slider   = widgets.IntSlider(value=0, min=0, max=99, description="随机种子")
    # n_slider      = widgets.IntSlider(value=6, min=1, max=24, description="样本数")
    # cols_slider   = widgets.IntSlider(value=3, min=1, max=6,  description="每行列数")
    # inv_check     = widgets.Checkbox(value=False, description="显示画面外点")
    # lbl_check     = widgets.Checkbox(value=True,  description="显示关节名")
    split_toggle  = widgets.ToggleButtons(options=["train", "val"], description="Split")
    seed_slider   = widgets.IntSlider(value=0, min=0, max=99, description="Seed")
    n_slider      = widgets.IntSlider(value=6, min=1, max=24, description="Samples")
    cols_slider   = widgets.IntSlider(value=3, min=1, max=6,  description="Cols")
    inv_check     = widgets.Checkbox(value=False, description="Show OOF pts")
    lbl_check     = widgets.Checkbox(value=True,  description="Show labels")
    ui = widgets.VBox([
        widgets.HBox([split_toggle, seed_slider]),
        widgets.HBox([n_slider, cols_slider, inv_check, lbl_check]),
    ])

    def _run(split, seed, n, cols, show_inv, show_lbl):
        json_path = train_json if split == "train" else val_json
        visualize_coco_samples(
            json_path=json_path,
            base_dir=base_dir,
            num_samples=n,
            cols=cols,
            random_seed=seed,
            show_invisible=show_inv,
            show_labels=show_lbl,
        )

    out = widgets.interactive_output(_run, {
        "split":    split_toggle,
        "seed":     seed_slider,
        "n":        n_slider,
        "cols":     cols_slider,
        "show_inv": inv_check,
        "show_lbl": lbl_check,
    })
    display(ui, out)



# 快速静态预览（train）
# visualize_coco_samples(TRAIN_JSON, BASE_DATA_DIR, num_samples=6)

# 交互式 widget
_make_validator_widget(BASE_DATA_DIR, TRAIN_JSON, VAL_JSON)

# 训练元数据定义
config

In [ ]:
from mmpose.datasets import CocoDataset

config_full_path = os.path.join(
    './mmpose/configs/body_2d_keypoint/rtmpose/coco',
    MODEL_CONFIG_NAME
)

if not os.path.exists(config_full_path):
    raise FileNotFoundError(f"未找到配置文件: {config_full_path}")

cfg = Config.fromfile(config_full_path)

mmpose_config_root = os.path.abspath('./mmpose/configs')
metainfo_file = os.path.join(mmpose_config_root, '_base_/datasets/coco.py')

if not os.path.exists(metainfo_file):
    raise FileNotFoundError(f"找不到 metainfo 文件: {metainfo_file}")
print(f"✓ Metainfo 文件已定位: {metainfo_file}")

cfg.work_dir  = WORK_DIR
cfg.device    = DEVICE
cfg.data_root = os.path.abspath(BASE_DATA_DIR)

for split, anno_file in [('train', train_anno_file), ('val', val_anno_file)]:
    dl = getattr(cfg, f'{split}_dataloader')
    dl.dataset.type       = 'CocoDataset'
    dl.dataset.data_root  = cfg.data_root
    dl.dataset.ann_file   = anno_file
    dl.dataset.data_prefix = dict(img='')
    dl.dataset.metainfo   = dict(from_file=metainfo_file)
    dl.batch_size         = BATCH_SIZE
    dl.num_workers        = min(4, os.cpu_count() // 2)

cfg.test_dataloader = cfg.val_dataloader

cfg.val_evaluator  = dict(type='CocoMetric', ann_file=val_anno_file, score_mode='keypoint')
cfg.test_evaluator = cfg.val_evaluator

cfg.optim_wrapper.optimizer.lr  = LR
cfg.train_cfg.max_epochs        = MAX_EPOCHS
cfg.train_cfg.val_interval      = VAL_INTERVAL

if 'param_scheduler' in cfg:
    for scheduler in cfg.param_scheduler:
        if scheduler.get('type') == 'CosineAnnealingLR':
            scheduler['T_max'] = MAX_EPOCHS
            scheduler['end']   = MAX_EPOCHS
        elif scheduler.get('type') == 'MultiStepLR':
            scheduler['milestones'] = [int(MAX_EPOCHS * 0.7), int(MAX_EPOCHS * 0.9)]

cfg.default_hooks.checkpoint.interval     = VAL_INTERVAL
cfg.default_hooks.checkpoint.max_keep_ckpts = 3
cfg.default_hooks.logger.interval          = 10

cfg.visualizer = dict(
    type='PoseLocalVisualizer',
    vis_backends=[dict(type='LocalVisBackend')],
    name='visualizer'
)

os.makedirs(WORK_DIR, exist_ok=True)
cfg.dump(os.path.join(WORK_DIR, 'actual_config.py'))

print(f"\n配置完成")
print(f"   Data Root: {cfg.data_root}")
print(f"   Train: {os.path.basename(train_anno_file)}")
print(f"   Val:   {os.path.basename(val_anno_file)}")

# 微调

In [ ]:
import warnings


In [ ]:
SIGMA = 6.0  #标签高斯平滑； set as default for RTMPose family


assert 'train_anno_file' in globals(), "请先运行 Cell 3 生成数据"

config_path = os.path.join('./mmpose/configs/body_2d_keypoint/rtmpose/coco',
                           'rtmpose-t_8xb256-420e_coco-256x192.py')
cfg = Config.fromfile(config_path)

mmpose_root = os.path.abspath('./mmpose/configs')
cfg.work_dir  = WORK_DIR
cfg.data_root = os.path.abspath(BASE_DATA_DIR)

train_pipeline = [
    dict(type='LoadImage', file_client_args=dict(backend='disk')),
    dict(type='GetBBoxCenterScale'),
    dict(type='RandomFlip', direction='horizontal', prob=0.5),
    dict(type='RandomBBoxTransform', scale_factor=[0.75, 1.25], shift_factor=0.1),
    dict(type='TopdownAffine', input_size=INPUT_SIZE, use_udp=True),
    dict(type='GenerateTarget', encoder=dict(
        type='SimCCLabel', input_size=INPUT_SIZE,
        smoothing_type='gaussian', sigma=SIGMA,
        simcc_split_ratio=2.0, label_smooth_weight=0.0,
    )),
    dict(type='PackPoseInputs')
]
test_pipeline = [
    dict(type='LoadImage', file_client_args=dict(backend='disk')),
    dict(type='GetBBoxCenterScale'),
    dict(type='TopdownAffine', input_size=INPUT_SIZE, use_udp=True),
    # dict(type='GenerateTarget', encoder=dict(
    #     type='SimCCLabel', input_size=INPUT_SIZE,
    #     smoothing_type='gaussian', sigma=6.0,
    #     simcc_split_ratio=2.0, label_smooth_weight=0.0,
    # )),
    dict(type='PackPoseInputs')
]

common_ds_kwargs = dict(
    data_root=cfg.data_root,
    data_prefix=dict(img=''),
    metainfo=dict(from_file=os.path.join(mmpose_root, '_base_/datasets/coco.py')),
)

cfg.train_dataloader = dict(
    batch_size=BATCH_SIZE, num_workers=4, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(type='CocoDataset', ann_file=train_anno_file, pipeline=train_pipeline, **common_ds_kwargs)
)
cfg.val_dataloader = dict(
    batch_size=BATCH_SIZE, num_workers=4, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type='CocoDataset', ann_file=val_anno_file, pipeline=test_pipeline, test_mode=True, **common_ds_kwargs)
)
cfg.test_dataloader = cfg.val_dataloader

cfg.val_evaluator  = dict(type='CocoMetric', ann_file=val_anno_file, score_mode='keypoint', nms_mode='none')
cfg.test_evaluator = cfg.val_evaluator

cfg.optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=LR, betas=(0.9, 0.999), weight_decay=0.01),
    paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True),
)
cfg.train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=MAX_EPOCHS, val_interval=VAL_INTERVAL)
cfg.param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(type='MultiStepLR', milestones=[int(MAX_EPOCHS*0.7), int(MAX_EPOCHS*0.9)], gamma=0.1),
]
cfg.default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=10),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=VAL_INTERVAL,
                    save_best='coco/AP', rule='greater', max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
)
cfg.visualizer = dict(type='PoseLocalVisualizer', vis_backends=[dict(type='LocalVisBackend')], name='visualizer')

# 修改模型头部为 SimCCLabel 编码；sigma==6.0， set as default for RTMPose family
cfg.model.head.decoder = dict(
    type='SimCCLabel', input_size=INPUT_SIZE,
    smoothing_type='gaussian', sigma=SIGMA, simcc_split_ratio=2.0,
)

os.makedirs(WORK_DIR, exist_ok=True)
cfg.dump(os.path.join(WORK_DIR, 'config.py'))

print(f"配置完成: RTMPose-tiny + SimCCLabel编码")
print(f"  输入尺寸: {INPUT_SIZE}")
print(f"  训练样本: {len(json.load(open(train_anno_file))['images'])}")


In [ ]:
import os
import sys
import contextlib
import functools
import warnings

def suppress_libpng_warnings(func):
    """
    装饰器：压制 libpng 的 C 层警告（如 eXIf: duplicate）。
    同时过滤 Python 层的 FutureWarning（torch.load weights_only）。
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Python 层警告过滤
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=".*eXIf.*")
            warnings.filterwarnings("ignore", category=FutureWarning, message=".*weights_only.*")

            # C 层 stderr 重定向：把 libpng 的输出丢到 /dev/null
            devnull_fd = os.open(os.devnull, os.O_WRONLY)
            old_stderr_fd = os.dup(2)          # 备份原始 stderr 文件描述符
            os.dup2(devnull_fd, 2)             # 把 fd=2 (stderr) 指向 /dev/null
            os.close(devnull_fd)

            try:
                result = func(*args, **kwargs)
            finally:
                os.dup2(old_stderr_fd, 2)      # 恢复 stderr
                os.close(old_stderr_fd)

        return result
    return wrapper

In [ ]:
runner = Runner.from_cfg(cfg)
print("\n start training.")

@suppress_libpng_warnings
runner.train()

## 微调指标展示

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# 最新完整训练的 scalars.json
scalars_path = Path(WORK_DIR) / "20260325_041645/vis_data/scalars.json"

with open(scalars_path) as f:
    lines = [json.loads(l) for l in f if l.strip()]

train_iters, train_loss, train_acc = [], [], []
val_epochs, val_ap = [], []

for entry in lines:
    if "loss" in entry and "coco/AP" not in entry:
        train_iters.append(entry.get("iter", len(train_iters)))
        train_loss.append(entry["loss"])
        if "acc_pose" in entry:
            train_acc.append(entry["acc_pose"])
    if "coco/AP" in entry:
        val_epochs.append(entry.get("epoch", len(val_epochs) * 5))
        val_ap.append(entry["coco/AP"])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(train_iters, train_loss, color="#2196F3", linewidth=1, alpha=0.8)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss (SimCC)")
axes[0].set_title("Train Loss")
axes[0].grid(alpha=0.3)

if train_acc:
    axes[1].plot(train_iters[:len(train_acc)], train_acc, color="#4CAF50", linewidth=1, alpha=0.8)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("acc_pose")
axes[1].set_title("Train Accuracy (per batch)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)

axes[2].plot(val_epochs, val_ap, color="#FF5722", marker="o", linewidth=2, markersize=6)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AP@0.5:0.95")
axes[2].set_title("Val AP (CocoMetric)")
axes[2].set_ylim(0, 1.05)
axes[2].grid(alpha=0.3)
for ep, ap in zip(val_epochs, val_ap):
    axes[2].annotate(f"{ap:.3f}", (ep, ap), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=8)

best_ap = max(val_ap)
best_ep = val_epochs[val_ap.index(best_ap)]
fig.suptitle(
    f"Training Curves  |  Best Val AP = {best_ap:.4f} @ epoch {best_ep}",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print(f"训练轮次: {len(set(val_epochs))} 次验证")
print(f"最终 loss: {train_loss[-1]:.6f}")
print(f"最佳 Val AP: {best_ap:.4f} @ epoch {best_ep}")

# 结果可视化 & 对比 Baseline
- 左侧：Baseline（微调前） 预测
- 右侧：Fine-tuned（微调后）预测
- 底部：AP/AR 曲线、关键点 PCK 对比柱状图

In [ ]:
# ============================================================
# 结果可视化 & 对比 Baseline
# 依赖：mmpose, torch, matplotlib, numpy, PIL, glob, pathlib
# ============================================================

import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from mmpose.apis import init_model, inference_topdown
from mmpose.utils import register_all_modules

register_all_modules()

# ── 路径配置 ────────────────────────────────────────────────
WORK_DIR_PATH = Path(WORK_DIR)
CONFIG_PATH   = str(WORK_DIR_PATH / "config.py")

# 自动定位最佳微调权重
_best_ckpts = sorted(WORK_DIR_PATH.glob("best_coco_AP_epoch_*.pth"))
if not _best_ckpts:
    _best_ckpts = sorted(WORK_DIR_PATH.glob("epoch_*.pth"))
if not _best_ckpts:
    raise FileNotFoundError(f"在 {WORK_DIR_PATH} 下未找到任何 checkpoint，请确认训练已完成")
FINETUNED_CKPT = str(_best_ckpts[-1])

# Baseline：用原始 RTMPose-tiny COCO 预训练权重（从官方 URL 或本地）
# 若本地没有，mmpose 会自动下载到 ~/.cache/mim/
BASELINE_CKPT = "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-tiny_simcc-coco_pt-aic-coco_420e-256x192-e613ba3f_20230127.pth"

NUM_COMPARE_SAMPLES = 6   # 对比样本数
COMPARE_COLS        = 2   # 固定2列（baseline | finetuned）
COMPARE_SEED        = 7

# ── 可视化颜色 ───────────────────────────────────────────────
SKEL_PALETTE = [
    "#FF6B6B","#FF6B6B","#4ECDC4","#4ECDC4","#45B7D1",
    "#96CEB4","#96CEB4","#FFEAA7","#DDA0DD","#DDA0DD",
    "#DDA0DD","#DDA0DD","#FFB347","#FFB347","#FFB347",
    "#87CEEB","#87CEEB","#87CEEB","#87CEEB",
]
SKEL_0IDX = [
    (15,13),(13,11),(16,14),(14,12),(11,12),
    (5,11),(6,12),(5,6),(5,7),(6,8),
    (7,9),(8,10),(1,2),(0,1),(0,2),
    (1,3),(2,4),(3,5),(4,6),
]
COCO17_NAMES = [
    "nose","l_eye","r_eye","l_ear","r_ear",
    "l_sho","r_sho","l_elb","r_elb","l_wri","r_wri",
    "l_hip","r_hip","l_kne","r_kne","l_ank","r_ank",
]

# ── 初始化两个模型 ───────────────────────────────────────────
print("加载 Baseline 模型...")
model_base = init_model(CONFIG_PATH, BASELINE_CKPT, device=DEVICE)
print("加载 Fine-tuned 模型...")
model_fine = init_model(CONFIG_PATH, FINETUNED_CKPT, device=DEVICE)
print("✓ 两个模型加载完成")


# ── 辅助：从推理结果提取关键点 ─────────────────────────────
def extract_kpts(result):
    """从 inference_topdown 返回的 PoseDataSample 提取 (17,3) 数组。"""
    if not result:
        return np.zeros((17, 3))
    kpts = result[0].pred_instances
    xy   = kpts.keypoints[0]         # (17, 2)
    sc   = kpts.keypoint_scores[0]   # (17,)
    return np.concatenate([xy, sc[:, None]], axis=1)  # (17, 3)


# ── 辅助：在 ax 上绘制骨骼叠加 ────────────────────────────
def _overlay_pose(ax, img_np, kpts_17x3, title, score_thr=0.3):
    ax.imshow(img_np)
    pts = kpts_17x3  # shape (17,3): x,y,score

    # 骨架连线
    for ei, (a, b) in enumerate(SKEL_0IDX):
        if pts[a, 2] > score_thr and pts[b, 2] > score_thr:
            color = SKEL_PALETTE[ei] if ei < len(SKEL_PALETTE) else "#FFFFFF"
            ax.plot([pts[a,0], pts[b,0]], [pts[a,1], pts[b,1]],
                    color=color, linewidth=2.5, zorder=3)

    # 关键点散点
    for i in range(17):
        if pts[i, 2] > score_thr:
            ax.scatter(pts[i,0], pts[i,1], s=50, color="#FF4500", zorder=5)

    ax.set_title(title, fontsize=10, pad=4)
    ax.axis("off")


# ── 从 val JSON 随机抽样 ─────────────────────────────────────
import json, random

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

random.seed(COMPARE_SEED)
sample_imgs = random.sample(val_coco["images"], min(NUM_COMPARE_SAMPLES, len(val_coco["images"])))
ann_by_imgid = {a["image_id"]: a for a in val_coco["annotations"]}


# ── 推理并收集结果 ───────────────────────────────────────────
results = []   # list of dict: img_np, kpts_base, kpts_fine, ann, img_info

print(f"\n对 {len(sample_imgs)} 张图像推理...")
for img_info in sample_imgs:
    img_path = str(Path(BASE_DATA_DIR) / img_info["file_name"])
    ann = ann_by_imgid.get(img_info["id"], {})
    bbox = ann.get("bbox", [0, 0, img_info["width"], img_info["height"]])
    # mmpose topdown 推理需要 bbox 格式 [x1,y1,x2,y2]
    x1, y1, w, h = bbox
    bboxes = np.array([[x1, y1, x1+w, y1+h, 1.0]])

    try:
        img_np = np.array(Image.open(img_path).convert("RGB"))
        r_base = inference_topdown(model_base, img_path, bboxes=bboxes, bbox_format='xyxy')
        r_fine = inference_topdown(model_fine, img_path, bboxes=bboxes, bbox_format='xyxy')
        kpts_base = extract_kpts(r_base)
        kpts_fine = extract_kpts(r_fine)
        results.append(dict(
            img_np=img_np,
            kpts_base=kpts_base,
            kpts_fine=kpts_fine,
            ann=ann,
            img_info=img_info,
        ))
    except Exception as e:
        print(f"  [跳过] {img_info['file_name']}: {e}")

print(f"✓ 推理完成，有效样本 {len(results)} 张")

if not results:
    print("无有效推理结果，检查图像路径or bbox格式")




In [ ]:
import json, numpy as np
from pathlib import Path

# 取 val 集第一张
with open(VAL_JSON) as f:
    val_coco = json.load(f)

img_info = val_coco["images"][0]
ann = {a["image_id"]: a for a in val_coco["annotations"]}[img_info["id"]]
bbox = ann["bbox"]
x1, y1, w, h = bbox
bboxes = np.array([[x1, y1, x1+w, y1+h]])

img_path = str(Path(BASE_DATA_DIR) / img_info["file_name"])
res = inference_topdown(model_fine, img_path, bboxes=bboxes, bbox_format='xyxy')

sample = res[0]
kpts = sample.pred_instances
print("keypoints shape     :", kpts.keypoints.shape)
print("keypoint_scores shape:", kpts.keypoint_scores.shape)

## 可视化对比
BUG

In [ ]:
# ── 并排对比可视化 ─────────────────────────────────────────
n = len(results)
fig, axes = plt.subplots(n, 2, figsize=(14, n * 5))
if n == 1:
    axes = axes[np.newaxis, :]  # 保证 2D 索引

for row, r in enumerate(results):
    fname = Path(r["img_info"]["file_name"]).name
    _overlay_pose(axes[row, 0], r["img_np"], r["kpts_base"],
                  f"Baseline  |  {fname}")
    _overlay_pose(axes[row, 1], r["img_np"], r["kpts_fine"],
                  f"Fine-tuned  |  {fname}")

# 列标题
axes[0, 0].set_title(f"◀ Baseline (pretrained COCO)\n{Path(results[0]['img_info']['file_name']).name}",
                     fontsize=11, color="#2196F3", pad=6)
axes[0, 1].set_title(f"Fine-tuned ▶\n{Path(results[0]['img_info']['file_name']).name}",
                     fontsize=11, color="#FF5722", pad=6)

fig.suptitle("Pose Estimation: Baseline vs Fine-tuned", fontsize=14, fontweight="bold", y=1.005)
plt.tight_layout()
plt.show()


# ── 定量指标对比：PCK@0.05 per keypoint ──────────────────────
# 用 val 集全部样本做快速 PCK 统计（不跑完整 CocoMetric，避免重复训练开销）

print("\n计算 PCK@0.05 (全 val 集)...")

SCORE_THR   = 0.3    # 推理置信度阈值
PCK_THR     = 0.05   # 以人体 torso 为归一化尺度

def compute_pck_val(model, val_coco, base_dir, thr=PCK_THR, score_thr=SCORE_THR, max_samples=200):
    """
    对 val 集（最多 max_samples 张）计算 per-joint PCK。
    归一化尺度 = max(bbox_w, bbox_h) * thr
    """
    ann_map = {a["image_id"]: a for a in val_coco["annotations"]}
    imgs    = val_coco["images"][:max_samples]
    hits    = np.zeros(17)
    totals  = np.zeros(17)

    for img_info in imgs:
        ann = ann_map.get(img_info["id"])
        if ann is None:
            continue
        bbox = ann.get("bbox", [0, 0, img_info["width"], img_info["height"]])
        x1, y1, w, h = bbox
        scale = max(w, h) * thr  # 归一化尺度
        if scale < 1:
            continue

        # bboxes = np.array([[x1, y1, x1+w, y1+h, 1.0]])
        bboxes = np.array([[x1, y1, x1+w, y1+h]])  # mmpose 2.0.0+ 需要无置信度bbox?（存疑
        img_path = str(Path(base_dir) / img_info["file_name"])
        try:
            res   = inference_topdown(model, img_path, bboxes=bboxes, bbox_format='xyxy')
            pred  = extract_kpts(res)  # (17, 3)
        except Exception:
            continue

        kps_flat = ann.get("keypoints", [])
        for j in range(17):
            gt_x  = kps_flat[j*3]
            gt_y  = kps_flat[j*3+1]
            gt_v  = kps_flat[j*3+2]
            if gt_v == 0:   # 画面外，跳过
                continue
            totals[j] += 1
            if pred[j, 2] < score_thr:
                continue
            dist = np.sqrt((pred[j,0]-gt_x)**2 + (pred[j,1]-gt_y)**2)
            if dist <= scale:
                hits[j] += 1

    pck = np.where(totals > 0, hits / totals, np.nan)
    return pck

pck_base = compute_pck_val(model_base, val_coco, BASE_DATA_DIR)
pck_fine = compute_pck_val(model_fine, val_coco, BASE_DATA_DIR)

mean_base = np.nanmean(pck_base)
mean_fine = np.nanmean(pck_fine)


# ── PCK 柱状图 ───────────────────────────────────────────────
x = np.arange(17)
width = 0.38

fig2, (ax_bar, ax_mean) = plt.subplots(
    1, 2, figsize=(16, 5),
    gridspec_kw=dict(width_ratios=[4, 1])
)

bars_b = ax_bar.bar(x - width/2, pck_base, width, label="Baseline",
                    color="#2196F3", alpha=0.85, edgecolor="white")
bars_f = ax_bar.bar(x + width/2, pck_fine, width, label="Fine-tuned",
                    color="#FF5722", alpha=0.85, edgecolor="white")

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(COCO17_NAMES, rotation=45, ha="right", fontsize=9)
ax_bar.set_ylabel("PCK@0.05", fontsize=11)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_title("Per-keypoint PCK@0.05", fontsize=12)
ax_bar.legend()
ax_bar.grid(axis="y", alpha=0.3)

# 标注改善量
for i in range(17):
    if not np.isnan(pck_base[i]) and not np.isnan(pck_fine[i]):
        delta = pck_fine[i] - pck_base[i]
        color = "#2E7D32" if delta >= 0 else "#C62828"
        ax_bar.text(i, max(pck_base[i], pck_fine[i]) + 0.02,
                    f"{delta:+.2f}", ha="center", va="bottom",
                    fontsize=7, color=color)

# 总体 mPCK 对比
ax_mean.bar(["Baseline", "Fine-tuned"], [mean_base, mean_fine],
            color=["#2196F3", "#FF5722"], alpha=0.9, edgecolor="white", width=0.5)
ax_mean.set_ylim(0, 1.1)
ax_mean.set_ylabel("mean PCK@0.05")
ax_mean.set_title("Overall mPCK", fontsize=12)
for i, (label, val) in enumerate(zip(["Baseline", "Fine-tuned"], [mean_base, mean_fine])):
    ax_mean.text(i, val + 0.02, f"{val:.3f}", ha="center", va="bottom",
                 fontsize=11, fontweight="bold")
delta_mean = mean_fine - mean_base
ax_mean.set_xlabel(
    f"Δ = {delta_mean:+.3f}  ({'↑ improved' if delta_mean >= 0 else '↓ degraded'})",
    fontsize=10,
    color="#2E7D32" if delta_mean >= 0 else "#C62828"
)
ax_mean.grid(axis="y", alpha=0.3)

fig2.suptitle(f"Quantitative Comparison  |  val samples: {min(200, len(val_coco['images']))}",
              fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n{'='*40}")
print(f"  Baseline  mPCK@0.05 : {mean_base:.4f}")
print(f"  Fine-tuned mPCK@0.05: {mean_fine:.4f}")
print(f"  Delta               : {delta_mean:+.4f}")
print(f"{'='*40}")